# Wielowymiarowa metoda Newtona — Ćwiczenie 4

Rozwiązujemy układ nieliniowych równań metodą Newtona w $\mathbb{R}^n$.

**Ogólny wzór:**
$$\mathbf{x}^{(i+1)} = \mathbf{x}^{(i)} - \mathbf{J}^{-1}\bigl(\mathbf{x}^{(i)}\bigr) \, \mathbf{f}\bigl(\mathbf{x}^{(i)}\bigr)$$

**Układ do rozwiązania:**
$$\begin{cases}
f_1(x_1, x_2) = x_1^2 - 4x_2^2 - 4 = 0 \\
f_2(x_1, x_2) = x_1^2 + x_2^2 - 9 = 0
\end{cases}$$

**Macierz Jacobiego:**
$$\mathbf{J}(x_1, x_2) = \begin{bmatrix}
\frac{\partial f_1}{\partial x_1} & \frac{\partial f_1}{\partial x_2} \\
\frac{\partial f_2}{\partial x_1} & \frac{\partial f_2}{\partial x_2}
\end{bmatrix}$$

Układ ma **4 pierwiastki**: $(\pm 2\sqrt{2},\, \pm 1)$.

> **Dowód:** Z $f_2=0$ mamy $x_1^2+x_2^2=9$. Z $f_1=0$ mamy $x_1^2-4x_2^2=4$. Odjmując: $5x_2^2=5 \Rightarrow x_2=\pm 1$, a wtedy $x_1^2=8 \Rightarrow x_1=\pm 2\sqrt{2}$.

In [ ]:
import numpy as np
from sympy import symbols, diff, Matrix, lambdify, pprint, init_printing, simplify
init_printing()

In [ ]:
x1, x2 = symbols('x1 x2')

f1_expr = x1**2 - 4*x2**2 - 4
f2_expr = x1**2 + x2**2 - 9

F_expr = Matrix([f1_expr, f2_expr])

print("Wektor funkcji f:")
pprint(F_expr)

J_expr = Matrix([
    [diff(f1_expr, x1), diff(f1_expr, x2)],
    [diff(f2_expr, x1), diff(f2_expr, x2)]
])

print("\nMacierz Jacobiego:")
pprint(J_expr)

print("\nOdwrócona macierz Jacobiego:")
pprint(J_expr.inv())

p1_expr = simplify(J_expr.inv() * F_expr)
print("\nWzór na krok Newtona: J^{{-1}} \u00b7 F:")
pprint(p1_expr)

In [ ]:
h1_func = lambdify((x1, x2), p1_expr[0], modules='numpy')
h2_func = lambdify((x1, x2), p1_expr[1], modules='numpy')

X = np.array([1.0, 0.8])
Xold = X.copy()
n_iter = 5
tol = 1e-6
errors = []

print("=== Newton (sympy symboliczny) ===")
print(f"Iteracja 0: x = [{X[0]:.6f}, {X[1]:.6f}]")

for i in range(n_iter):
    step = np.array([float(h1_func(X[0], X[1])), float(h2_func(X[0], X[1]))])
    Xnew = X - step
    err = np.linalg.norm(Xnew - Xold)
    errors.append(err)
    Xold = Xnew.copy()
    X = Xnew.copy()
    print(f"Iteracja {i+1}: x = [{X[0]:.6f}, {X[1]:.6f}], b\u0142\u0105d = {err:.2e}")
    if err < tol:
        print(f"Zbie\u017cno\u015b\u00f3\u0107 osi\u0105gni\u0119ta po {i+1} iteracjach.")
        break

print(f"\nRozwi\u0105zanie: x = [{X[0]:.6f}, {X[1]:.6f}]")

## Implementacja numeryczna

Zamiast symbolicznego wyprowadzania pochodnych, obliczamy macierz Jacobiego i krok Newtona w całości numerycznie, korzystając z `numpy`.

Warto użyć `np.linalg.solve(J, b)` zamiast `np.linalg.inv(J) @ b`, ponieważ jest to stabilniejsze numerycznie i szybsze.

---

**ĆWICZENIE 1:** Znajdź wszystkie 4 pierwiastki układu, zmieniając punkt startowy.

**ĆWICZENIE 2:** Zaimplementuj krok Newtona jawnie jako `dX = -np.linalg.inv(J(x)) @ F(x)` (zamiast `np.linalg.solve`). Porównaj wyniki.

In [ ]:
f1 = lambda x: x[0]**2 - 4*x[1]**2 - 4
f2 = lambda x: x[0]**2 + x[1]**2 - 9
F_vec = lambda x: np.array([f1(x), f2(x)])

def J_numeric(x):
    return np.array([
        [2*x[0], -8*x[1]],
        [2*x[0],  2*x[1]]
    ])

X = np.array([1.0, 0.8])
Xold = X.copy()
errors_num = []

print("=== Newton (numeryczny, np.linalg.solve) ===")
print(f"Iteracja 0: x = [{X[0]:.6f}, {X[1]:.6f}]")

for i in range(n_iter):
    step = np.linalg.solve(J_numeric(X), -F_vec(X))
    Xold = X.copy()
    X = X + step
    err = np.linalg.norm(X - Xold)
    errors_num.append(err)
    print(f"Iteracja {i+1}: x = [{X[0]:.6f}, {X[1]:.6f}], b\u0142\u0105d = {err:.2e}")
    if err < tol:
        print(f"Zbie\u017cno\u015b\u00f3\u0107 osi\u0105gni\u0119ta po {i+1} iteracjach.")
        break

print(f"\nRozwi\u0105zanie: x = [{X[0]:.6f}, {X[1]:.6f}]")
print(f"Sprawdzenie: f(X) = [{F_vec(X)[0]:.2e}, {F_vec(X)[1]:.2e}]")

## Ćwiczenia do samodzielnego wykonania

### Ćwiczenie 1 — Znalezienie wszystkich pierwiastków

Układ ma 4 pierwiastki: $(+2\sqrt{2},\,+1)$, $(+2\sqrt{2},\,-1)$, $(-2\sqrt{2},\,+1)$, $(-2\sqrt{2},\,-1)$.

Uruchom numeryczną metodę Newtona z różnymi punktami startowymi (np. `[1.0, 0.8]`, `[1.0, -1.0]`, `[-1.0, 1.0]`, `[-1.0, -0.5]`) i sprawdź, czy algorytm zbiega do wszystkich czterech rozwiązań.

| Punkt startowy | Znaleziony pierwiastek | # iteracji |
|---|---|---|
| `(1.0, 0.8)` | ... | ... |
| `(1.0, -1.0)` | ... | ... |
| `(-1.0, 1.0)` | ... | ... |
| `(-1.0, -0.5)` | ... | ... |

---

### Ćwiczenie 2 — Jawne odwracanie macierzy Jacobiego

W poniższej komórce zaimplementuj krok Newtona, jawnie obliczając odwrotność macierzy Jacobiego przez `np.linalg.inv()`. Porównaj wyniki z wersją używającą `np.linalg.solve()`.

**Pytanie refleksyjne:** Czy `np.linalg.solve()` zawsze daje lepsze wyniki? Kiedy warto użyć `np.linalg.inv()`?

In [ ]:
# ĆWICZENIE 2 — Implementacja z jawną odwrotnością Jacobianu

X = np.array([1.0, 0.8])
Xold = X.copy()
errors_inv = []

print("=== Newton (numeryczny, np.linalg.inv) ===")
print(f"Iteracja 0: x = [{X[0]:.6f}, {X[1]:.6f}]")

for i in range(n_iter):
    # TODO: Oblicz odwrotność macierzy Jacobiego
    J_inv = np.linalg.inv(J_numeric(X))

    # TODO: Oblicz wektor funkcji
    F = F_vec(X)

    # TODO: Oblicz krok jako J_inv @ (-F)
    step = J_inv @ (-F)

    Xold = X.copy()
    X = X + step
    err = np.linalg.norm(X - Xold)
    errors_inv.append(err)
    print(f"Iteracja {i+1}: x = [{X[0]:.6f}, {X[1]:.6f}], b\u0142\u0105d = {err:.2e}")
    if err < tol:
        print(f"Zbie\u017cno\u015b\u00f3\u0107 osi\u0105gni\u0119ta po {i+1} iteracjach.")
        break

print(f"\nRozwi\u0105zanie: x = [{X[0]:.6f}, {X[1]:.6f}]")

## Podsumowanie

| Funkcja | Opis | Stabilność |
|---|---|---|
| `np.linalg.solve(A, b)` | Rozwiązuje $Ax=b$ (bez odwracania $A$) | Wysoka (LU decomposition) |
| `np.linalg.inv(A)` | Oblicza $A^{-1}$ jawnie | Niższa (czuła na conditioning) |
| `sympy.lambdify` | Konwertuje wyrażenie symboliczne na funkcję numeryczną | -- |

**Wielowymiarowa metoda Newtona:**
- Szybka zbieżność kwadratowa w pobliżu pierwiastka
- Wymaga obliczania macierzy Jacobiego
- Punkt startowy ma duże znaczenie — różne obszary zbieżności dla różnych pierwiastków